### Кластеризация
# Задача
Кластеризация покупателей для выявления покупателей со схожими характеристиками

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
%matplotlib inline

Загружаем таблицы

In [30]:
orders = pd.read_csv("../dataset/olist_orders_dataset.csv")
order_items = pd.read_csv("../dataset/olist_order_items_dataset.csv")
customers = pd.read_csv("../dataset/olist_customers_dataset.csv")
products = pd.read_csv("../dataset/olist_products_dataset.csv")
sellers = pd.read_csv("../dataset/olist_sellers_dataset.csv")
order_reviews = pd.read_csv("../dataset/olist_order_reviews_dataset.csv")
order_payments = pd.read_csv("../dataset/olist_order_payments_dataset.csv")
product_translation = pd.read_csv("../dataset/product_category_name_translation.csv")

Перевод названий продуктов на английский

In [31]:
products = products.merge(product_translation, on='product_category_name', how='left')

Объединяем таблицы

In [32]:
order_items_full = order_items.merge(products, on='product_id', how='left')
order_items_full = order_items_full.merge(sellers, on='seller_id', how='left')

In [33]:
orders_full = orders.merge(order_items_full, on='order_id', how='left')
orders_full = orders_full.merge(customers, on='customer_id', how='left')
orders_full = orders_full.merge(order_reviews[['order_id','review_score']], on='order_id', how='left')

In [ ]:
payments_agg = order_payments.groupby('order_id').agg({
    'payment_value':'sum',
    'payment_type': lambda x: x.mode()[0]
}).reset_index()
orders_full = orders_full.merge(payments_agg, on='order_id', how='left')

In [ ]:
print(orders_full.info())

In [ ]:
orders_full.head()

In [ ]:
# Преобразование временных меток
orders_full['order_purchase_timestamp'] = pd.to_datetime(
    orders_full['order_purchase_timestamp'], errors='coerce'
)
orders_full['order_approved_at'] = pd.to_datetime(
    orders_full['order_approved_at'], errors='coerce'
)
orders_full['order_delivered_carrier_date'] = pd.to_datetime(
    orders_full['order_delivered_carrier_date'], errors='coerce'
)
orders_full['order_delivered_customer_date'] = pd.to_datetime(
    orders_full['order_delivered_customer_date'], errors='coerce'
)

In [ ]:
# Извлечение временных признаков
orders_full['purchase_hour'] = orders_full['order_purchase_timestamp'].dt.hour
orders_full['purchase_dayofweek'] = orders_full['order_purchase_timestamp'].dt.dayofweek
orders_full['purchase_month'] = orders_full['order_purchase_timestamp'].dt.month

In [ ]:
# Расчет времени доставки
orders_full['delivery_time_days'] = (
    orders_full['order_delivered_customer_date'] - orders_full['order_purchase_timestamp']
).dt.total_seconds() / (24 * 3600)

In [ ]:
# Время утверждения заказа
orders_full['approval_time_hours'] = (
    orders_full['order_approved_at'] - orders_full['order_purchase_timestamp']
).dt.total_seconds() / 3600

In [ ]:
orders_full['approval_time_hours'].fillna(orders_full['approval_time_hours'].median(), inplace=True)

orders_full['delivery_time_days'].fillna(orders_full['delivery_time_days'].median(), inplace=True)

orders_full['order_approved_at'].fillna(orders_full['order_approved_at'].median(), inplace=True)

orders_full['order_delivered_carrier_date'].fillna(orders_full['order_delivered_carrier_date'].median(), inplace=True)

orders_full['order_delivered_customer_date'].fillna(orders_full['order_delivered_customer_date'].median(), inplace=True)

Агрегация признаков по клиентам

In [ ]:
customer_features = orders_full.groupby('customer_id').agg({
    'price': 'mean',
    'freight_value': 'mean',
    'delivery_time_days': 'mean',
    'order_id': 'nunique',
    'review_score': 'mean'
}).rename(columns={
    'price': 'avg_price',
    'freight_value': 'avg_freight',
    'delivery_time_days': 'avg_delivery_time',
    'order_id': 'num_orders',
    'review_score': 'avg_review'
}).reset_index()

In [ ]:
# Удаляем пропуски
customer_features = customer_features.dropna()

In [ ]:
print("Форма данных после агрегации:", customer_features.shape)
customer_features.describe()

Масштабирование признаков

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(customer_features[['avg_price', 'avg_freight', 'avg_delivery_time', 'num_orders', 'avg_review']])

### Кластеризация методом K-средних

In [ ]:
inertia = []
K = range(2, 11)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(6,4))
plt.plot(K, inertia, 'o-')
plt.xlabel('Количество кластеров')
plt.ylabel('Inertia')
plt.title('Метод локтя для подбора числа кластеров')
plt.show()

Опитимально k = 5

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
customer_features['cluster_kmeans'] = kmeans.fit_predict(X)

Анализ кластеров KMeans

In [ ]:
cluster_summary = customer_features.groupby('cluster_kmeans').mean().round(2)
print("Средние значения признаков по кластерам KMeans:")
display(cluster_summary)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=customer_features['cluster_kmeans'], cmap='viridis', alpha=0.7)
plt.title('Кластеры покупателей (KMeans)')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.show()

Кластеризация методом DBSCAN

In [ ]:
dbscan = DBSCAN(eps=0.8, min_samples=10)
customer_features['cluster_dbscan'] = dbscan.fit_predict(X)

print("Количество уникальных кластеров (DBSCAN):", customer_features['cluster_dbscan'].nunique())

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=customer_features['cluster_dbscan'], cmap='plasma', alpha=0.7)
plt.title('Кластеры покупателей (DBSCAN)')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.show()

Иерархическая кластеризация (Agglomerative)

In [ ]:
agg = AgglomerativeClustering(n_clusters=4)
customer_features['cluster_agg'] = agg.fit_predict(X)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=customer_features['cluster_agg'], cmap='cool', alpha=0.7)
plt.title('Кластеры покупателей (Agglomerative Clustering)')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.show()

Сравнение результатов

In [ ]:
print("Сравнение распределения клиентов по кластерам:")
print("\nKMeans:")
print(customer_features['cluster_kmeans'].value_counts())

print("\nDBSCAN:")
print(customer_features['cluster_dbscan'].value_counts())

print("\nAgglomerative:")
print(customer_features['cluster_agg'].value_counts())

Интерпретация кластеров (KMeans как основной)

In [ ]:
cluster_summary = customer_features.groupby('cluster_kmeans').agg({
    'avg_price': 'mean',
    'avg_freight': 'mean',
    'avg_delivery_time': 'mean',
    'num_orders': 'mean',
    'avg_review': 'mean'
}).round(2)

print("\nИтоговая таблица характеристик кластеров (KMeans):")
display(cluster_summary)